Load Dataset

In [10]:
import pandas as pd
import numpy as np

# Load the COX-2 dataset
df = pd.read_csv('cox2.csv')

# Determine target column
target_column = 'cox2Class' if 'cox2Class' in df.columns else df.columns[-1]

# Separate features and target, explicitly dropping data-leaking biological metrics
# We add 'IC50' to the drop list so the model only looks at molecular descriptors
leakage_columns = [target_column, 'IC50']
# Use errors='ignore' in case there are slight variations in spelling
X = df.drop(columns=[col for col in leakage_columns if col in df.columns], errors='ignore')
y = df[target_column]

print(f"Identified target column: '{target_column}'")
print(f"Dataset loaded successfully: {X.shape[0]} compounds, {X.shape[1]} structural molecular features.")

Identified target column: 'cox2Class'
Dataset loaded successfully: 462 compounds, 255 structural molecular features.


Split the dataset

In [11]:
from sklearn.model_selection import train_test_split

# Split the dataset into 75% training and 25% test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Training set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

Training set: 346 samples | Test set: 116 samples


Select a learning method

In [12]:
from sklearn.ensemble import RandomForestClassifier

# Select the RandomForestClassifier for the classification task
rf_clf = RandomForestClassifier(random_state=42)

Define a tuning grid

In [13]:
# 'max_features' serves as the scikit-learn equivalent to the 'mtry' parameter in R
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20],
    'max_features': ['sqrt', 'log2', 0.3, 0.5]
}

Perform 10-fold cross-validation

In [14]:
from sklearn.model_selection import GridSearchCV
import warnings

# Suppress runtime multiprocessing warnings if they occur
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Set up GridSearch with 10-fold cross-validation evaluating classification Accuracy
grid_search = GridSearchCV(
    estimator=rf_clf, 
    param_grid=param_grid, 
    cv=10, 
    scoring='accuracy', 
    n_jobs=-1
)

# Execute the hyperparameter search grid on the training data
grid_search.fit(X_train, y_train)

# Isolate the best performing tuned model
best_model = grid_search.best_estimator_

Analyze performance values

In [15]:
# Print cross-validation accuracy results
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"10-Fold CV Accuracy: {grid_search.best_score_:.4f}\n")

# Extract and rank the molecular feature importances
importances = best_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Molecular Descriptor': X.columns, 
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("Top 10 Most Important Features for COX-2 Inhibition:")
print(feat_imp_df.head(10).to_string(index=False))

Best Hyperparameters: {'max_depth': 10, 'max_features': 'log2', 'n_estimators': 150}
10-Fold CV Accuracy: 0.8616

Top 10 Most Important Features for COX-2 Inhibition:
Molecular Descriptor  Importance
      QikProp_QPlogS    0.019734
     QikProp_accptHB    0.017062
   QikProp_QPlogPo.w    0.015668
   QikProp_QPlogKhsa    0.015009
     moe2D_logP.o.w.    0.014533
          moe2D_logS    0.011947
     QikProp_QPlogKp    0.011572
   moe2D_GCUT_PEOE_2    0.011511
moe2D_PEOE_VSA_FPPOS    0.011278
      moeGao_chi4pcv    0.010932


Apply the final model to the test set & Evaluate performance

In [16]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predict categories for the unseen test compounds
y_pred = best_model.predict(X_test)

# Calculate classification validation metrics
test_acc = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Test Set Accuracy: {test_acc:.4f}\n")
print("Confusion Matrix:")
print(conf_matrix)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

Test Set Accuracy: 0.7759

Confusion Matrix:
[[ 6 20]
 [ 6 84]]

Detailed Classification Report:
              precision    recall  f1-score   support

      Active       0.50      0.23      0.32        26
    Inactive       0.81      0.93      0.87        90

    accuracy                           0.78       116
   macro avg       0.65      0.58      0.59       116
weighted avg       0.74      0.78      0.74       116

